# 02 · Collect Docker Q&A from Stack Exchange (Stack Overflow + sister sites)

Pulls **accepted-answer** questions tagged `docker` (plus `docker-compose` / `dockerfile` on Stack Overflow)
through the official Stack Exchange API. Every row is a `(problem, accepted answer)` pair that a community
member marked as the fix.

* **Free, no login.** Without a key the API allows ~300 requests/day per IP. A free key raises that to 10,000/day:
  create one at <https://stackapps.com/apps/oauth/register>, then add it in Colab (key icon, left sidebar) as a secret named `SE_KEY`. Optional but recommended.
* **Quota.** A full Stack Overflow run needs roughly 250 requests: easy with a key, right at the limit without one (it stops cleanly and you re-run tomorrow).
* **Resumable.** Everything fetched is cached under `raw_sources/stackexchange/cache/`. If Colab stops or the quota runs out, run the notebook again and it continues.
* **Licence.** Content is CC BY-SA 4.0 (older posts 3.0). Each row keeps its `url` and `license` so attribution is possible.
* No usernames are stored.

Output: `raw_sources/stackexchange/<site>.jsonl`, consumed by notebook 05.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
# window_months: the API returns at most 25 pages x 100 questions per query, so we slice by date
# and take the top-voted questions in each slice. per_window is derived from max_questions.
SITES = {
    "stackoverflow": {"tags": ["docker", "docker-compose", "dockerfile"], "max_questions": 15000, "window_months": 3},
    "serverfault":   {"tags": ["docker"], "max_questions": 1500, "window_months": 12},
    "superuser":     {"tags": ["docker"], "max_questions": 600,  "window_months": 12},
    "askubuntu":     {"tags": ["docker"], "max_questions": 400,  "window_months": 12},
    "unix":          {"tags": ["docker"], "max_questions": 800,  "window_months": 12},
    "devops":        {"tags": ["docker"], "max_questions": 500,  "window_months": 12},
}
MIN_QUESTION_SCORE = 2     # drop questions almost nobody found useful
FROM_YEAR = 2014           # Docker before this is too different from today
PROBLEM_ONLY = True        # keep only questions that read like a problem (error text, "fails", ...)
USE_DRIVE = True

!pip install -q beautifulsoup4 requests

In [ ]:
import json
import math
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = Path("/content/drive/MyDrive/customer-support-triage/tickets")
    except ImportError:
        OUT_ROOT = Path("tickets")
else:
    OUT_ROOT = Path("tickets")

SE_DIR = OUT_ROOT / "raw_sources" / "stackexchange"
CACHE_DIR = SE_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_secret(name):
    # Colab secrets first, then environment variables. The value is never printed.
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


SE_KEY = get_secret("SE_KEY")
print("Writing to:", OUT_ROOT.resolve(), "| API key:", "yes" if SE_KEY else "no (300 requests/day limit)")

## 1 · API helper

Respects the API's `backoff` instruction and stops cleanly when the daily quota is nearly used up.

In [ ]:
API = "https://api.stackexchange.com/2.3"
session = requests.Session()


class QuotaExhausted(Exception):
    pass


def se_get(path, **params):
    params.setdefault("pagesize", 100)
    if SE_KEY:
        params["key"] = SE_KEY
    for attempt in range(4):
        resp = session.get(f"{API}/{path}", params=params, timeout=60)
        if resp.status_code in (502, 503, 429):
            time.sleep(5 * (attempt + 1))
            continue
        resp.raise_for_status()
        data = resp.json()
        if data.get("backoff"):
            time.sleep(data["backoff"] + 1)
        if data.get("quota_remaining", 100) < 3:
            raise QuotaExhausted("Daily quota used up - re-run this notebook tomorrow (or add SE_KEY).")
        return data
    raise RuntimeError(f"Stack Exchange API kept failing for {path}")

## 2 · HTML → readable text

Posts come back as HTML. Code blocks become fenced blocks, inline code keeps its backticks, images are dropped.

In [ ]:
BLOCK_TAGS = ["p", "div", "h1", "h2", "h3", "h4", "h5", "h6", "ul", "ol", "blockquote"]


def html_to_text(html_text):
    soup = BeautifulSoup(html_text or "", "html.parser")
    for pre in soup.find_all("pre"):
        pre.replace_with("\n```\n" + pre.get_text().strip("\n") + "\n```\n")
    for code in soup.find_all("code"):
        code.replace_with("`" + code.get_text() + "`")
    for tag in soup.find_all(["img", "script", "style"]):
        tag.decompose()
    for link in soup.find_all("a"):
        href, text = link.get("href", ""), link.get_text()
        link.replace_with(f"{text} ({href})" if href.startswith("http") and text.strip() != href else text)
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for li in soup.find_all("li"):
        li.insert(0, "- ")
        li.append("\n")
    for block in soup.find_all(BLOCK_TAGS):
        block.append("\n\n")
    text = soup.get_text("")
    return re.sub(r"\n{3,}", "\n\n", text).strip()


print(html_to_text("<p>Run <code>docker ps</code>:</p><pre><code>$ docker ps\nCONTAINER</code></pre><ul><li>one</li><li>two</li></ul>"))

## 3 · Problem filter

Top-voted Docker questions are often conceptual ("what is the difference between…"). When `PROBLEM_ONLY` is on we keep
only questions whose title or body reads like a failure, so we don't spend quota fetching answers to questions that are not tickets.

In [ ]:
TITLE_TROUBLE_RE = re.compile(
    r"\b(error|errors|fail(s|ed|ing|ure)?|cannot|can't|couldn't|unable|not working|doesn't work|does not work|"
    r"won't|denied|refused|time(d)? ?out|exception|crash(es|ed)?|stuck|hangs?|exits?|exited|not found|no such|"
    r"unauthorized|unhealthy|invalid|unexpected|missing|conflict|why (is|does|do|am)|no space)\b", re.I)
BODY_TROUBLE_RE = re.compile(r"\b(error|exception|denied|refused|failed|cannot|traceback|no such)\b", re.I)


def looks_like_problem(title, body_text):
    return bool(TITLE_TROUBLE_RE.search(title)) or bool(BODY_TROUBLE_RE.search(body_text[:3000]))

## 4 · Fetch questions (date-sliced, cached)

In [ ]:
def month_windows(from_year, months):
    start = datetime(from_year, 1, 1, tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)
    windows = []
    while start < now:
        month = start.month - 1 + months
        end = datetime(start.year + month // 12, month % 12 + 1, 1, tzinfo=timezone.utc)
        windows.append((int(start.timestamp()), int(end.timestamp()) - 1))
        start = end
    return windows


QUESTION_FIELDS = ["question_id", "title", "body", "tags", "score", "accepted_answer_id",
                   "creation_date", "link", "view_count", "answer_count", "content_license"]


def fetch_window(site, tag, from_ts, to_ts, per_window, exclude_tag=None):
    # Cache format: {"items": [...], "exhausted": bool}. "exhausted" means the API had no more
    # pages for this window, so it's safe to reuse even if a later run asks for a higher per_window.
    # (Older cache files are a plain list - treated as not-yet-exhausted, so a higher target tops them up.)
    cache_file = CACHE_DIR / f"q__{site}__{tag}__{from_ts}.json"
    if cache_file.exists():
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        cached_items, exhausted = (cached["items"], cached["exhausted"]) if isinstance(cached, dict) else (cached, False)
        if exhausted or len(cached_items) >= per_window:
            return cached_items[:per_window]

    items, page, exhausted = [], 1, False
    while len(items) < per_window and page <= 25:
        params = dict(site=site, tagged=tag, accepted="True", sort="votes", order="desc",
                      min=MIN_QUESTION_SCORE, fromdate=from_ts, todate=to_ts,
                      page=page, filter="withbody")
        if exclude_tag:
            params["nottagged"] = exclude_tag
        data = se_get("search/advanced", **params)
        for q in data["items"]:
            items.append({k: q.get(k) for k in QUESTION_FIELDS})
        if not data.get("has_more"):
            exhausted = True
            break
        page += 1

    items = items[:per_window]
    cache_file.write_text(json.dumps({"items": items, "exhausted": exhausted}), encoding="utf-8")
    return items


def collect_questions(site, cfg):
    windows = month_windows(FROM_YEAR, cfg["window_months"])
    per_window = max(100, math.ceil(cfg["max_questions"] / len(windows) / 100) * 100)
    found = {}
    for tag in cfg["tags"]:
        exclude = "docker" if tag != "docker" else None   # avoid re-fetching questions already found via the docker tag
        for from_ts, to_ts in windows:
            for q in fetch_window(site, tag, from_ts, to_ts, per_window, exclude):
                found.setdefault(q["question_id"], q)
        print(f"  {site}/{tag}: {len(found)} questions so far")
    return list(found.values())

## 5 · Fetch the accepted answers and write rows

In [ ]:
def fetch_answers(site, answer_ids):
    cache_file = CACHE_DIR / f"a__{site}.jsonl"
    cached = {}
    if cache_file.exists():
        for line in cache_file.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            cached[row["answer_id"]] = row

    missing = [i for i in answer_ids if i not in cached]
    with cache_file.open("a", encoding="utf-8") as out:
        for start in range(0, len(missing), 100):
            batch = missing[start:start + 100]
            data = se_get(f"answers/{';'.join(map(str, batch))}", site=site, filter="withbody")
            for a in data["items"]:
                row = {"answer_id": a["answer_id"], "body": a["body"], "score": a["score"],
                       "content_license": a.get("content_license")}
                cached[row["answer_id"]] = row
                out.write(json.dumps(row) + "\n")
    return cached


def iso(ts):
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()


def build_rows(site, questions, answers):
    source_type = "stackoverflow_accepted" if site == "stackoverflow" else "stackexchange_accepted"
    rows = []
    for q in questions:
        ans = answers.get(q["accepted_answer_id"])
        if not ans:
            continue
        rows.append({
            "source_id": f"{site}:{q['question_id']}",
            "source": site,
            "source_type": source_type,
            "url": q["link"],
            "license": q["content_license"] or ans.get("content_license") or "CC BY-SA",
            "created_at": iso(q["creation_date"]),
            "title": q["title"],
            "problem": q["_problem"],
            "resolution": html_to_text(ans["body"]),
            "resolution_kind": "accepted_answer",
            "problem_score": q["score"],
            "resolution_score": ans["score"],
            "tags": q["tags"],
            "extra": {"view_count": q["view_count"], "answer_count": q["answer_count"]},
        })
    return rows

In [ ]:
stats = {}
try:
    for site, cfg in SITES.items():
        print(f"== {site}")
        questions = collect_questions(site, cfg)
        for q in questions:
            q["title"] = html_to_text(q["title"])
            q["_problem"] = html_to_text(q["body"])
        kept = [q for q in questions if q["accepted_answer_id"] and
                (not PROBLEM_ONLY or looks_like_problem(q["title"], q["_problem"]))]
        answers = fetch_answers(site, [q["accepted_answer_id"] for q in kept])
        rows = build_rows(site, kept, answers)

        SE_DIR.mkdir(parents=True, exist_ok=True)
        with (SE_DIR / f"{site}.jsonl").open("w", encoding="utf-8") as out:
            for row in rows:
                out.write(json.dumps(row, ensure_ascii=False) + "\n")
        stats[site] = {"questions_fetched": len(questions), "problem_like": len(kept), "rows_written": len(rows)}
        print("  ", stats[site])
except QuotaExhausted as exc:
    print("STOPPED:", exc)

(SE_DIR / "collect_report.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
print(json.dumps(stats, indent=2))

## 6 · Spot-check

In [ ]:
import random

for site in stats:
    path = SE_DIR / f"{site}.jsonl"
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
    print(f"\n=== {site} ({len(rows)} rows)")
    for row in random.sample(rows, min(2, len(rows))):
        print(f"[{row['created_at'][:10]}  q:{row['problem_score']}  a:{row['resolution_score']}] {row['title']}")
        print("    ->", row["resolution"][:180].replace("\n", " "))